# Escalar más allá de un ordenador

El mismo circuito y el mismo código se pueden ejecutar en el clúster de CESGA cambiando solo el valor de `infrastructure`. La mayoría de quienes sigan esta serie no tendrán acceso a un clúster con SLURM ni a hardware cuántico real: por eso los ejemplos de este notebook se limitan a la validación de argumentos, que sí ocurre sin necesitar ese acceso, en vez de a una ejecución real.

## CUNQA: QPUs distribuidas y simuladas

`infrastructure="cunqa"` ejecuta contra la plataforma CUNQA de CESGA, que reparte QPUs simuladas entre varios nodos gestionados por SLURM. Se necesita instalar el paquete `cunqa` aparte, que no viene incluido con `polypus-quantum`; se explica en el apartado [Installation](https://github.com/Bahia-Software/polypus#installation) del README principal.

Tres argumentos con nombre adicionales dimensionan la reserva de SLURM, con efecto solo en esta infraestructura:

- `nodes`: número de nodos de SLURM a reservar. Por defecto 1.
- `cores_per_qpu`: núcleos de CPU por cada QPU. Por defecto 2.
- `walltime`: walltime de SLURM, el tiempo máximo que las QPUs quedan reservadas, en formato `HH:MM:SS`. Por defecto `"10:00:00"`.

Una llamada real se vería así:

```python
result = polypus.run_quantum_circuit(
    qc,
    shots=1000,
    infrastructure="cunqa",
    n_qpus=10,
    nodes=2,
    cores_per_qpu=4,
    walltime="02:00:00",
)
```

`nodes` y `cores_per_qpu` deben ser al menos 1, y se comprueba antes de intentar llegar a SLURM, así que el error se ve sin tener CUNQA instalado:

In [ ]:
import polypus

bell = polypus.Circuit(2).h(0).cx(0, 1).measure_all()

try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="cunqa", nodes=0)
except ValueError as e:
    print(e)

`walltime` solo acepta el formato `HH:MM:SS`: horas con una o más cifras, así que `"48:00:00"` es válido, y minutos y segundos con dos cifras por debajo de 60. El formato de SLURM con días, `D-HH:MM:SS`, se rechaza, también antes de llegar a SLURM:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", walltime="2-00:00:00"
    )
except ValueError as e:
    print(e)

Cuando se agota el walltime, las QPUs se liberan, así que para un entrenamiento largo conviene fijarlo por encima de la duración esperada. Polypus ayuda con eso: al entrenar en CUNQA, emite un aviso tras la primera generación si, a ese ritmo, la ejecución fuera a durar más que el walltime.

Las QPUs de CUNQA simulan con Aer, así que con `infrastructure="cunqa"` el `backend` tiene que ser Aer, el `"aer"` por defecto. Cualquier otro valor se rechaza en vez de ignorarse en silencio:

In [ ]:
try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", backend="polypus"
    )
except ValueError as e:
    print(e)

Tampoco se acepta un `noise_model`, una descripción de Qiskit Aer de los errores de una máquina real que el backend Aer local sí puede simular: las QPUs de CUNQA se configuran al reservarlas, no en cada ejecución:

In [ ]:
from qiskit_aer.noise import NoiseModel

try:
    polypus.run_quantum_circuit(
        bell, shots=1000, infrastructure="cunqa", noise_model=NoiseModel()
    )
except ValueError as e:
    print(e)

## QMIO: hardware cuántico real

`infrastructure="qmio"` ejecuta directamente contra QMIO, el ordenador cuántico real de CESGA, a través de un cliente en Rust puro. El paquete publicado en PyPI no incluye este backend: hace falta compilar Polypus desde el código fuente con la característica `qmio` activada, así que el intento falla con un mensaje claro:

In [ ]:
try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="qmio")
except ValueError as e:
    print(e)

Por ser hardware real, QMIO tampoco acepta una semilla explícita: el ruido de la propia máquina no se puede fijar como el de un simulador, y `seed` se rechaza igual de pronto:

In [ ]:
try:
    polypus.run_quantum_circuit(bell, shots=1000, infrastructure="qmio", seed=42)
except ValueError as e:
    print(e)

Una compilación con la característica `qmio` necesita además saber dónde está QMIO: no hay dirección por defecto. Se lee de la variable de entorno `ZMQ_SERVER`, o se pasa con `options={"endpoint": ...}`, y sin ninguna de las dos la ejecución falla con un `polypus.QmioError`.

## Resumen

Este notebook ha cubierto cómo el mismo código se dirige a CUNQA o a QMIO cambiando `infrastructure`, los argumentos adicionales de cada una, y sus validaciones reales, vistas sin necesitar acceso a un clúster ni a hardware cuántico.

La siguiente sección retoma circuitos concretos: teletransporte cuántico y la transformada de Fourier cuántica.

## Siguiente paso

Continúa con: [`04a_teletransporte.ipynb`](04a_teletransporte.ipynb).